# Model ENSO fullclim 12-2025

December 2022, January 2023, July 2024, November 2025

Caroline Juang, c.juang@columbia.edu

**After model training on `Model_ENSOclim_Akaike` , we can use this file to input gSST and output our estimates of counterfactual climate. This file is used to look at gSST influence on any of our climate variables, not just on the burned area-specific ones.**


**Inputs:**
* SST gradient (1982-present) - build it following the README
* trained models `Model_ENSOclim_Akaike` (SST gradient -> climate)
* trained models `Model_Akaike` (climate -> burned area)

**Variables**
* `sstpred` generally refers to the detrended SST gradient, which is the detrended-SST gradient scenario put into the climate->burned area model.
* `climpred` generally refers to the observed SST gradient, put into the climate->burned area model.

**Outputs:**
* Predicted climate (1983-present) (for all climate variables, not just for the burned area-relevant ones).


This version uses the same as `Model_ENSOfull12-2022` but only predicts the **counterfactual climate scenario under detrended gSST**. This way, we can use any counterfactual climate as an input into any climate-BA model (great for the cross-validation step).

The model is based on the outputs of `Model_ENSOclim_Akaike` and `Model_Akaike`.

Data source:
* NOAA sea surface temperature, https://psl.noaa.gov/data/gridded/data.noaa.ersst.v5.html

save and load models

Machine Learning Mastery: https://machinelearningmastery.com/save-load-machine-learning-models-python-scikit-learn/

In [1]:
# import

from customconfig import *
from customscripts import *

import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr
import joblib

In [2]:
# customize seasons for climate variables

# customize number of rolling periods
time_length = int(finalyear-firstyear+1) # get length of timeseries
# translate years into dates
firsttime = str(firstyear)+'-01-01'
finaltime = str(finalyear)+'-12-31'

# Read the climate index from the .txt file
with open('0_climindname.txt', 'r') as f:
    climindname = f.read().strip()
print(climindname +' will be used for the SST gradient')
climindnameDT = 'DeTrend_'+climindname
climfilename = 'DeTrendClimObs_'+climindname # sst-predicted BA

# importing data string
data_string = 'data//'
model_string = 'model//'+climindname+'//'
predict_string = 'predicted//'+climindname+'//'

# folder for saving figures
figfolder = 'your_fig_folder' # customize this

patch125-155_nino3-34 will be used for the SST gradient


## Import burned area data
from `Data_CreateModelData`

In [3]:
# import observed burned area
filename = data_string + 'WUMI-ecoprovinces'
wumi = pd.read_csv(filename+'_all.txt').set_index('Unnamed: 0')
wumifor = pd.read_csv(filename+'_for.txt').set_index('Unnamed: 0')
wuminon = pd.read_csv(filename+'_non.txt').set_index('Unnamed: 0')
print('imported '+filename+' all, for, non')

imported data//WUMI-ecoprovinces all, for, non


## Import observed SST gradient data
from `Data_CreateENSOIndex` and `Data_CreateModelData`

In [4]:
# import observed SST gradient
filename = data_string + 'sstgrad_seasons_' + climindname+'_82_y.txt'
climind82_seasons = pd.read_csv(filename).set_index('Unnamed: 0')
print('imported '+filename)

# import de-trended SST gradient
filename = data_string + 'sstgrad_seasons_' + climindnameDT+'_82_y.txt'
climind82_seasonsDT = pd.read_csv(filename).set_index('Unnamed: 0')
print('imported '+filename)

# import observed SST gradient (with extra year, for comparison)
filename = data_string + 'sstgrad_seasons_' + climindname+'_81_y.txt'
climind81_seasons = pd.read_csv(filename).set_index('Unnamed: 0')
print('imported '+filename)

# IMPORT gSST AVERAGE (average of concurrent-season and prior-season)
# import observed SST gradient
filename = data_string + 'sstgrad_seasons_' + climindname+'_82_y_avgcurr-prior.txt'
climind82_seasonsavg = pd.read_csv(filename).set_index('Unnamed: 0')
print('imported '+filename)

# import de-trended SST gradient
filename = data_string + 'sstgrad_seasons_' + climindnameDT+'_82_y_avgcurr-prior.txt'
climind82_seasonsDTavg = pd.read_csv(filename).set_index('Unnamed: 0')
print('imported '+filename)

# put the dataframes together
climind82_seasons = pd.concat([climind82_seasons, climind82_seasonsavg], axis=1)
climind82_seasonsDT = pd.concat([climind82_seasonsDT, climind82_seasonsDTavg], axis=1)

imported data//sstgrad_seasons_patch125-155_nino3-34_82_y.txt
imported data//sstgrad_seasons_DeTrend_patch125-155_nino3-34_82_y.txt
imported data//sstgrad_seasons_patch125-155_nino3-34_81_y.txt
imported data//sstgrad_seasons_patch125-155_nino3-34_82_y_avgcurr-prior.txt
imported data//sstgrad_seasons_DeTrend_patch125-155_nino3-34_82_y_avgcurr-prior.txt


## Import observed climate data
from `Data_CreateModelData`

In [5]:
# import OBSERVED climate data to feed into models
# climate is predicted using SST, but this is the climate observed data to check

dfframesall = {}
dfframesfor = {}
dfframesnon = {}

climind = pd.read_csv(data_string + 'gradient_'+climindname+'.txt', header=None, sep=",", skiprows=[0]).set_index(0)
for iecoreg in np.arange(len(province_num)):
    filename = data_string + 'climate_ecoprovinces_'
    print(dfnames[iecoreg])
    dfframesall[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_all.txt').set_index('Unnamed: 0')
    dfframesfor[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_for.txt').set_index('Unnamed: 0')
    dfframesnon[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_non.txt').set_index('Unnamed: 0')

allwestUS
ecoprov1
ecoprov2
ecoprov3
ecoprov4
ecoprov5
ecoprov6
ecoprov7
ecoprov8
ecoprov9
ecoprov10
ecoprov12
ecoprov13
ecoprov14
ecoprov15
ecoprov16
ecoprov17
ecoprov18
ecoprov19
ecoprov20
ecoprov21


## Import model outputs
from `Model_ENSOclim_Akaike` and `Model_Akaike`

In [6]:
# get the model input variable names from the txt files
# sst gradient to predict climate variables

with open(model_string + "modeloutput_sst_all_all.txt", "r") as f:
    inputsall = [line.strip() for line in f]
f.close()
with open(model_string + "modeloutput_sst_for.txt", "r") as f:
    inputsfor = [line.strip() for line in f]
f.close()
with open(model_string + "modeloutput_sst_non.txt", "r") as f:
    inputsnon = [line.strip() for line in f]
f.close()

# get indices of where each ecoregion's outputs begin
iinputsall = [i for i, e in 
               enumerate(inputsall) if "+++" in e]
# get ecoregions so iteration is not manual
iinputsfor = [i for i, e in 
               enumerate(inputsfor) if "+++" in e]
# get ecoregions so iteration is not manual
iinputsnon = [i for i, e in 
               enumerate(inputsnon) if "+++" in e]
# add in last index
iinputsall.append(len(inputsall)+1)
iinputsfor.append(len(inputsfor)+1)
iinputsnon.append(len(inputsnon)+1)

In [7]:
# get the model input variable names from the txt files
# climate variables to predict burned area

with open(model_string + "modeloutput_burnarea_for.txt", "r") as f:
    inputsfor_clim = [line.strip() for line in f]
f.close()
with open(model_string + "modeloutput_burnarea_non.txt", "r") as f:
    inputsnon_clim = [line.strip() for line in f]
f.close()

# get indices of where each ecoregion's outputs begin
# get ecoregions so iteration is not manual
iinputsfor_clim = [i for i, e in 
               enumerate(inputsfor_clim) if "+++" in e]
# get ecoregions so iteration is not manual
iinputsnon_clim = [i for i, e in 
               enumerate(inputsnon_clim) if "+++" in e]
# add in last index
iinputsfor_clim.append(len(inputsfor_clim)+1)
iinputsnon_clim.append(len(inputsnon_clim)+1)

# Setup for exporting burned area predictions
Predicted climate
* SST_obs-predicted climate
* SST_neg-predicted climate
* difference between climate_sstobs and climate_sstneg, called climate_preddiff
* observed climate with climate_preddiff subtracted from it, called climate_sstdiff

Predicted burned area
* SST-predicted burned area (observed SST, goes through SST-clim and clim-BA models)
* climate-predicted burned area (observed climate, goes through clim-BA model)

In [8]:
# a bunch of dictionaries for storage
# predicted climate from observed SST
dfsstpredclimall_obsSST = {}
dfsstpredclimfor_obsSST = {}
dfsstpredclimnon_obsSST = {}

# predicted climate from negative-trended SST
dfsstpredclimall_negSST = {}
dfsstpredclimfor_negSST = {}
dfsstpredclimnon_negSST = {}

# difference in predicted climate scenarios, climate_preddiff
dfsstpredclimall_preddiff = {}
dfsstpredclimfor_preddiff = {}
dfsstpredclimnon_preddiff = {}

# observed climate minus climate_preddiff
dfsstpredclimall_sstdiff = {}
dfsstpredclimfor_sstdiff = {}
dfsstpredclimnon_sstdiff = {}

In [9]:
# a bunch of dictionaries for storage

# SST-predicted, from climate_sstdiff
dfsstpredBAall = []
dfsstpredBAfor = []
dfsstpredBAnon = []
# climate-predicted, from observed climate
dfclimpredBAall = []
dfclimpredBAfor = []
dfclimpredBAnon = []

# scripts

In [10]:
# get the SST inputs needed
# MODIFIED to get every single variable!!!! (11/17/2025)
def modelinputsst_string(inputlist, varname, istart, iend):
    """
    Requirements: 
    inputlist = list of the Model_ENSOclim_Akaike model results (locally defined)
    ithisecoreg = the ecoregion name (globally defined)
    ithisecoregend = the next ecoregion name 
        in the list (globally defined)
    varname = the climate variable name (locally defined)
    """
    # narrow inputs list to ecoregion
    modellist = inputlist[istart:iend+1]
    # get all PREDICTING (where inputs start)
    iinputs = [i for i, e in 
               enumerate(modellist) if "PREDICTING" in e]
    # extract model inputs
    istart = modellist.index('PREDICTING ' + varname)
    if len(iinputs)>((iinputs.index(istart))+1): # 
        iend = iinputs[(iinputs.index(istart))+1]
        modelinputs = modellist[istart+1:iend] # range in list
    else:
        modelinputs = modellist[istart+1:] # last predictor to end of list
    return modelinputs

In [11]:
# get the SST inputs needed
def sortInputStringsMOD(inputlist, istart, iend):
    """
    Requirements: 
    inputlist = list of the Model_ENSOclim_Akaike model results (locally defined)
    istart = the ecoregion name
    iend = the next ecoregion name in the list
    """
    # narrow inputs list to ecoregion
    #modellistsort = inputlist[istart+1:iend+1]
    
    # MOD: RUN ALL CLIMATE VARIABLES, 
    # NOT JUST A SELECTION (11/17/2025)
    modellistsort = dfframesfor[ecoregname].columns.values
    
    # separate concurrent and previous-year variables
    modely0names = [s for s in modellistsort if "y0" in s]
    modely1names = [s for s in modellistsort if "y-1" in s]
    return [modellistsort, modely0names, modely1names]

In [12]:
# printing significance
def addSigMarker(pvalue):
    """
    Add an asterisk for values that are significant.
    p<0.05, add *
    p<0.01, add **
    otherwise add an empty space
    """
    if pvalue<0.01:
        adddot = '**' # significance marker
    elif pvalue<0.05:
        adddot = '*'# add significance marker
    else:
        adddot = ' ' # no significance
    return adddot

In [13]:
def predictClimateMOD(SSTgradient):
    """
    The goal is to predict all the climate variables used in the climate->burned
    area model, using our SST gradient->climate model, and put them in a 
    digestible format for inclusion in the climate->BA model/
    Inputs: seasonal variables 1983-present, in pandas DataFrame
        Plus requirements--which specify land type and ecoregion.
    Output: predictors for climate variable, in pandas DataFrame
    
    Requirements (a lot of variables):
        landname = should be 'all', 'for', or 'non'
    SST GRADIENT->CLIMATE MODEL: from imported TXT string of model variables.
        landtypeinput = Variable names, divided by ecoregion. For one land type.
        landtypeiinput = Locations (by line indices) of the start of variable names.
        ithisecoreg = Index of where ecoregion starts in list
        ithisecoregend = Index of where ecoregion ends in list
    CLIMATE->BURNED AREA MODEL: from imported TXT string of model variables.
        landtypeinputclim = Variable names, divided by ecoregion. For one land type.
        landtypeiinputclim = Locations (by line indices) of the start of variable names.
        ithisecoreg_clim = index of where ecoregion starts in list
        ithisecoregend_clim = index of where ecoregion ends in list
    sortInputStrings = script that intakes landtypeinputclim and outputs the same
        list but sorted into three buckets of the entire list (sorted, 
        y0 names, and y-1 names)
    """
    # MOD to remove all the file writing (11/17/2025)
    
    ecoregindex = dfnames.index(ecoregname)
    print('\nModeling the climate for '+landname+' in '+prov_abbr_names[ecoregindex])

    # single-out the variables needed for the two models
    ithisecoreg = landtypeinput.index('+++'+ecoregname)
    ithisecoregend = landtypeiinput[landtypeiinput.index(ithisecoreg)+1]-1
    ithisecoreg_clim = landtypeinputclim.index('+++'+ecoregname)
    ithisecoregend_clim = landtypeiinputclim[landtypeiinputclim.index(ithisecoreg_clim)+1]-1
    [climnamessort, climy0names, climy1names] = sortInputStringsMOD(landtypeinputclim,
                                                                ithisecoreg_clim, 
                                                                ithisecoregend_clim)
    
    # storage
    climmodeldict = {} # the loaded model
    climpred = [] # predicted climate variable values
    climnames = [] # name of climate variable
    
    # open models
    if len(climy0names)>0:
        for thisname in climy0names:
            filename = model_string + 'model_'+landname+'_'+thisname+'_'+ecoregname+'.sav'
            #print('\tLoading model: '+ filename)
            climmodeldict[thisname] = joblib.load(filename)
            # retrieve models
            modelinputs = modelinputsst_string(landtypeinput, thisname,
                                              ithisecoreg,
                                              ithisecoregend)
            #print('\tModel inputs: '+', '.join(modelinputs))
            # format the variable name for write to file
            tmpfullname, thisunits, thiscolor = suppclimplotformat(thisname)
            # get the column(s) of SST
            if len(modelinputs)==1:
                tmpX = np.asarray(SSTgradient[modelinputs]).reshape(-1,1)
            else:
                tmpX = np.asarray(SSTgradient[modelinputs])
            # predict climate, choose timeframe 1984 to present
            tmppred = climmodeldict[thisname].predict(tmpX).flatten()[1:].tolist()
            print('\t{:}_predicted = '.format(tmpfullname))

            for thisvari, thisvarname in enumerate(modelinputs):
                tmpfullvarname, thisunits, thiscolor = suppclimplotformat('gsst '+thisvarname)
                print('\t+ {:.3f}({:})'.format(climmodeldict[thisname].coef_.flatten()[thisvari], tmpfullvarname))

            print('\t+ {:.3f}'.format(climmodeldict[thisname].intercept_.item()))

            climpred.append(tmppred)
            climnames.append(thisname)
            # correlation observed vs. predicted area
            if landname=='for':
                tmpobsclimate = dfframesfor[ecoregname][thisname]
            if landname=='non':
                tmpobsclimate = dfframesnon[ecoregname][thisname]
            tmpr = pearsonr(tmpobsclimate, np.array(tmppred).flatten())
            tmppmarker = addSigMarker(tmpr.pvalue)
            print('\t\tr-value(obs clim, pred clim): {:.2}{:}'.format(tmpr.statistic, 
                                                                      tmppmarker))
    if len(climy1names)>0:
        for thisname in climy1names:
            tmpfullname, thisunits, thiscolor = suppclimplotformat(thisname)
            filename = model_string + 'model_'+landname+'_'+thisname+'_'+ecoregname+'.sav'
            filename = filename.replace("y-1", "y0") # change to get model name
            tmpname = thisname.replace("y-1", "y0")
            #print('\tLoading model: '+ filename)
            climmodeldict[thisname] = joblib.load(filename)
            # retrieve models
            modelinputs = modelinputsst_string(landtypeinput, tmpname,
                                               ithisecoreg,
                                               ithisecoregend)
            #print('\tModel inputs: '+' '.join(modelinputs))
            # get the column(s) of SST
            if len(modelinputs)==1:
                tmpX = np.asarray(SSTgradient[modelinputs]).reshape(-1,1)
            else:
                tmpX = np.asarray(SSTgradient[modelinputs])
            # predict climate, choose timeframe 1983 to prior-year
            tmppred = climmodeldict[thisname].predict(tmpX).flatten()[:-1].tolist()
            print('\t{:}_predicted = '.format(tmpfullname))
           
            print('\t\t+ {:.3f}'.format(climmodeldict[thisname].intercept_.item()))

            climpred.append(tmppred)
            climnames.append(thisname)
            # correlation observed vs. predicted area
            if landname=='for':
                tmpobsclimate = dfframesfor[ecoregname][thisname]
            if landname=='non':
                tmpobsclimate = dfframesnon[ecoregname][thisname]
            tmpr = pearsonr(tmpobsclimate, np.array(tmppred).flatten())
            tmppmarker = addSigMarker(tmpr.pvalue)
            print('\tr-value(obs clim, pred clim): {:.2}'.format(tmpr.statistic, tmppmarker))

    # create pandas dataframe to feed into prediction, sort
    climpred_T = np.array(climpred).transpose()
    Xclimpred = pd.DataFrame(climpred_T, index=None, columns=climnames).reindex(climnamessort, axis=1)    
    return Xclimpred

# SST -> climate

## observed SST by ecoregion

In [14]:
# write the same print statements to this file

# get this ecoregion
for ecoregname in dfnames:
    # FOREST import relevant models
    landname = 'for'
    landtypeinput = inputsfor
    landtypeiinput = iinputsfor
    landtypeinputclim = inputsfor_clim
    landtypeiinputclim = iinputsfor_clim

    # open models and predict climate
    Xclimpred = predictClimateMOD(climind82_seasons)

    # save to dict
    dfsstpredclimfor_obsSST[ecoregname] = Xclimpred

    # NONFOREST import relevant models
    landname = 'non'
    landtypeinput = inputsnon
    landtypeiinput = iinputsnon
    landtypeinputclim = inputsnon_clim
    landtypeiinputclim = iinputsnon_clim

    # open models and predict climate
    Xclimpred = predictClimateMOD(climind82_seasons)

    # save to dict
    dfsstpredclimnon_obsSST[ecoregname] = Xclimpred


Modeling the climate for for in 0: All western US
	Precipitation y-0,JFM_predicted = 
	+ -0.204(gSST y-0,JFM)
	+ 0.517
		r-value(obs clim, pred clim): 0.1 
	Precipitation y-0,AMJ_predicted = 
	+ -0.127(gSST y-0,AMJ)
	+ 0.282
		r-value(obs clim, pred clim): 0.11 
	Precipitation y-0,JAS_predicted = 
	+ -0.174(gSST y-0,JAS)
	+ 0.547
		r-value(obs clim, pred clim): 0.15 
	Precipitation y-0,OND_predicted = 
	+ 0.139(gSST y-0,OND)
	+ -0.547
		r-value(obs clim, pred clim): 0.17 
	RH y-0,JFM_predicted = 
	+ -0.529(gSST y-0,JFM)
	+ 1.339
		r-value(obs clim, pred clim): 0.51**
	RH y-0,AMJ_predicted = 
	+ -0.610(gSST y-0,1-6)
	+ 1.449
		r-value(obs clim, pred clim): 0.5**
	RH y-0,JAS_predicted = 
	+ 0.000(gSST y-0,JAS)
	+ -0.000
		r-value(obs clim, pred clim): nan 
	RH y-0,OND_predicted = 
	+ -0.162(gSST y-0,OND)
	+ 0.636
		r-value(obs clim, pred clim): 0.23 
	Solar_radiation y-0,JFM_predicted = 
	+ 0.295(gSST y-0,JFM)
	+ -0.746
		r-value(obs clim, pred clim): 0.17 
	Solar_radiation y-0,AMJ_pred

/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


ValueError: 'PREDICTING tmean y0 mo 1-3' is not in list

## de-trended SST by ecoregion

In [ ]:
for ecoregname in dfnames:

    # FOREST import relevant models
    landname = 'for'
    landtypeinput = inputsfor
    landtypeiinput = iinputsfor
    landtypeinputclim = inputsfor_clim
    landtypeiinputclim = iinputsfor_clim

    # open models and predict climate
    Xclimpred = predictClimateMOD(climind82_seasonsDT)

    # save to dict
    dfsstpredclimfor_negSST[ecoregname] = Xclimpred

    # NONFOREST import relevant models
    landname = 'non'
    landtypeinput = inputsnon
    landtypeiinput = iinputsnon
    landtypeinputclim = inputsnon_clim
    landtypeiinputclim = iinputsnon_clim

    # open models and predict climate
    Xclimpred = predictClimateMOD(climind82_seasonsDT)

    # save to dict
    dfsstpredclimnon_negSST[ecoregname] = Xclimpred

	Solar_radiation y-0,OND_predicted = 
	+ 0.053(gSST y-0,OND)
	+ -0.201
		r-value(obs clim, pred clim): 0.1 
	Tmax y-0,JFM_predicted = 
	+ -0.053(gSST y-0,JFM)
	+ 0.131
		r-value(obs clim, pred clim): 0.14 
	Tmax y-0,AMJ_predicted = 
	+ 0.348(gSST y-0,AMJ)
	+ -0.741
		r-value(obs clim, pred clim): 0.098 
	Tmax y-0,JAS_predicted = 
	+ 0.483(gSST y-0,4-9)
	+ -1.229
		r-value(obs clim, pred clim): 0.2 
	Tmax y-0,OND_predicted = 
	+ 0.284(gSST y-0,OND)
	+ -1.074
		r-value(obs clim, pred clim): 0.28 
	Tmin y-0,JFM_predicted = 
	+ -0.189(gSST y-1,10-3)
	+ 0.578
		r-value(obs clim, pred clim): 0.24 
	Tmin y-0,AMJ_predicted = 
	+ 0.126(gSST y-0,AMJ)
	+ -0.268
		r-value(obs clim, pred clim): -0.079 
	Tmin y-0,JAS_predicted = 
	+ 0.251(gSST y-0,JAS)
	+ -0.741
		r-value(obs clim, pred clim): 0.099 
	Tmin y-0,OND_predicted = 
	+ 0.227(gSST y-0,OND)
	+ -0.858
		r-value(obs clim, pred clim): 0.2 
	Tmean y-0,JFM_predicted = 
	+ -0.123(gSST y-0,JFM)
	+ 0.301
		r-value(obs clim, pred clim): 0.18 


	Tmean y-0,AMJ_predicted = 
	+ 0.275(gSST y-0,AMJ)
	+ -0.586
		r-value(obs clim, pred clim): 0.037 
	Tmean y-0,JAS_predicted = 
	+ 0.446(gSST y-0,4-9)
	+ -1.133
		r-value(obs clim, pred clim): 0.17 
	Tmean y-0,OND_predicted = 
	+ 0.267(gSST y-0,OND)
	+ -1.009
		r-value(obs clim, pred clim): 0.25 
	VPD y-0,JFM_predicted = 
	+ 0.241(gSST y-0,JFM)
	+ -0.590
		r-value(obs clim, pred clim): 0.11 
	VPD y-0,AMJ_predicted = 
	+ 0.522(gSST y-0,1-6)
	+ -1.198
		r-value(obs clim, pred clim): 0.28 
	VPD y-0,JAS_predicted = 
	+ 0.523(gSST y-0,4-9)
	+ -1.330
		r-value(obs clim, pred clim): 0.23 
	VPD y-0,OND_predicted = 
	+ 0.278(gSST y-0,OND)
	+ -1.050
		r-value(obs clim, pred clim): 0.31 
	Wet_days y-0,JFM_predicted = 
	+ -0.034(gSST y-0,JFM)
	+ 0.083
		r-value(obs clim, pred clim): -0.041 
	Wet_days y-0,AMJ_predicted = 
	+ -0.510(gSST y-0,AMJ)
	+ 1.087
		r-value(obs clim, pred clim): 0.28 
	Wet_days y-0,JAS_predicted = 
	+ -0.191(gSST y-0,JAS)
	+ 0.563
		r-value(obs clim, pred clim): 0.085 
	Wet_

/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))
/Users/caroline/Documents/GitHub/Project_ENSOFires_2022/Experimental Code/scikithere/lib/python3.11/site-packages/scipy/stats/_stats_py.

## Get difference between predictions

Difference in climate variable predictions. Both are SST-predicted, one is from the observed SST gradient and the other is from the negatively-trended SST gradient.

In [ ]:
# iterate through each ecoregion
for thisname in dfnames:
    # FOREST
    # get this ecoregion's predicted climate
    tmpsstobsclim = dfsstpredclimfor_obsSST[thisname]
    tmpsstnegclim = dfsstpredclimfor_negSST[thisname]
    tmpcols = tmpsstobsclim.columns
    tmpobsclim = dfframesfor[thisname][tmpcols]
    # take difference and add to dict
    tmppreddiff = tmpsstobsclim - tmpsstnegclim # diff in predictions
    tmpdiff = tmpobsclim - tmppreddiff # diff from obs
    dfsstpredclimfor_sstdiff[thisname] = tmpdiff
    
    # NONFOREST
    # get this ecoregion's predicted climate
    tmpsstobsclim = dfsstpredclimnon_obsSST[thisname]
    tmpsstnegclim = dfsstpredclimnon_negSST[thisname]
    tmpcols = tmpsstobsclim.columns
    tmpobsclim = dfframesnon[thisname][tmpcols]
    # take difference and add to dict
    tmppreddiff = tmpsstobsclim - tmpsstnegclim # diff in predictions
    tmpdiff = tmpobsclim - tmppreddiff # diff from obs
    dfsstpredclimnon_sstdiff[thisname] = tmpdiff

In [ ]:
# EXPORT climate variables without the SST gradient trend

for thisname in dfnames:
    # rearrange for pandas df
    tmpoutputfor = dfsstpredclimfor_sstdiff[thisname]
    tmpoutputnon = dfsstpredclimnon_sstdiff[thisname]

    dictoutputs = {'sstpred_for':tmpoutputfor, 'sstpred_non':tmpoutputnon}
    dictoutputnames = ['sstpred_for', 'sstpred_non']

    # convert to pandas df and export
    for i,thisdict in enumerate(dictoutputs):
        filename = predict_string + 'Climate_'+thisname+'_'+dictoutputnames[i]+'_'+climfilename+'_'+str(firstyear)+'-'+str(finalyear)+'.txt'
        dictoutputs[dictoutputnames[i]].to_csv(filename)

        print('Exported: '+filename)

Exported: predicted//patch125-155_nino3-34d//Climate_ecoprov7_sstpred_non_DeTrendClimObs_patch125-155_nino3-34d_1984-2022.txt
Exported: predicted//patch125-155_nino3-34d//Climate_ecoprov8_sstpred_for_DeTrendClimObs_patch125-155_nino3-34d_1984-2022.txt
Exported: predicted//patch125-155_nino3-34d//Climate_ecoprov8_sstpred_non_DeTrendClimObs_patch125-155_nino3-34d_1984-2022.txt
Exported: predicted//patch125-155_nino3-34d//Climate_ecoprov9_sstpred_for_DeTrendClimObs_patch125-155_nino3-34d_1984-2022.txt
Exported: predicted//patch125-155_nino3-34d//Climate_ecoprov9_sstpred_non_DeTrendClimObs_patch125-155_nino3-34d_1984-2022.txt
Exported: predicted//patch125-155_nino3-34d//Climate_ecoprov10_sstpred_for_DeTrendClimObs_patch125-155_nino3-34d_1984-2022.txt
Exported: predicted//patch125-155_nino3-34d//Climate_ecoprov10_sstpred_non_DeTrendClimObs_patch125-155_nino3-34d_1984-2022.txt
Exported: predicted//patch125-155_nino3-34d//Climate_ecoprov12_sstpred_for_DeTrendClimObs_patch125-155_nino3-34d_198

In [ ]:
# datetime object containing current date and time
from datetime import datetime
now = datetime.now()
# dd/mm/YY H:M:S
dt_string = now.strftime("%d/%m/%Y %H:%M:%S")
print("Model last run =", dt_string)